# Latent-Space Noise Injection: Mechanism, Effect, and Theory

Visualizes the latent-space noise injection used during DGD training (`training.latent_noise_enabled`/`_start`/`_end`), reading a completed run's saved checkpoints -- **no training happens here**.

**Mechanism:** before the decoder sees $z$, isotropic noise is added: $\tilde z = z + \epsilon,\ \epsilon \sim \mathcal{N}(0, \sigma_t^2 I)$. `noise_start`/`noise_end` (annealed via `cosine_noise_schedule`) configure a *target expected displacement* $D_t$ -- dimension-independent, so $\mathbb{E}[\|\epsilon\|] \approx D_t$ whether the latent is 2D or 20D -- not the per-coordinate std $\sigma_t$ directly. The actual per-coordinate std drawn each step is $\sigma_t = D_t/\sqrt{d}$ ($d$ = `representation.n_features`, see `noise_std_for_dim` in `src/utils/schedules.py`). The decoder only ever trains against $\tilde z$, never clean $z$.

**Why:** the DGD has no encoder or reparameterization trick -- $z$ is optimized directly via MAP, so nothing forces the decoder to behave sensibly off the exact training points. Training against a noisy neighborhood of each $z_i$ pushes the decoder toward a smooth mapping (the same principle as denoising autoencoders), while annealing $D_t \to 0$ lets representations settle into precise, non-overlapping positions the GMM prior can separate.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra import initialize, compose

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import ConvDecoder
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration
from src.utils.checkpoint import load_checkpoint
from src.utils.schedules import noise_schedule
from src.data import create_dataloaders, collect_all_labels, pick_class_representative_indices
from src.visualization import plot_noise_comparison, plot_noise_optimization_shift

device = setup_device(verbose=True)
set_random_seed(seed=42, device=device)
setup_cuml_acceleration(verbose=True)


In [ ]:
with initialize(version_base=None, config_path="../config"):
    config = compose(config_name="config")

config.paths.experiments_dir = str(project_root / "experiments")

experiments_dir = Path(config.paths.experiments_dir)
candidates = sorted(
    p for p in experiments_dir.glob(f"*_{config.experiment_name}")
    if (p / "models" / "best").is_dir()
)
assert candidates, (
    f"No completed training runs found under {experiments_dir} matching "
    f"*_{config.experiment_name} (looked for a models/best/ subfolder). "
    "Run dgd_training_demo.ipynb first."
)
run_dir = candidates[-1]  # newest, since the timestamp prefix sorts lexicographically
print(f"Using training run: {run_dir}")

# This run's own saved config, not the live config/config.yaml -- so the
# decoder architecture and noise schedule below always match what this run
# was actually trained with, even if config.yaml has changed since.
trained_cfg = OmegaConf.load(run_dir / "config.yaml")


In [ ]:
def decoder_factory():
    return ConvDecoder(
        latent_dim=trained_cfg.model.representation.n_features,
        hidden_dims=trained_cfg.model.decoder.hidden_dims,
        output_channels=trained_cfg.model.decoder.output_channels,
        output_size=trained_cfg.model.decoder.output_size,
        activation=trained_cfg.model.decoder.activation,
        final_activation=trained_cfg.model.decoder.final_activation,
        dropout_rate=trained_cfg.model.decoder.dropout_rate,
        init_size=trained_cfg.model.decoder.init_size,
    )

checkpoint_dirs = sorted(
    (run_dir / "models" / "checkpoints").glob("epoch_*"),
    key=lambda p: int(p.name.split('_')[1]),
)
checkpoint_epochs = [int(p.name.split('_')[1]) for p in checkpoint_dirs]
print(f"Found {len(checkpoint_dirs)} saved checkpoints at epochs: {checkpoint_epochs}")

noise_start = trained_cfg.training.get('latent_noise_start', 1.0)
noise_end = trained_cfg.training.get('latent_noise_end', 0.01)
noise_enabled = trained_cfg.training.get('latent_noise_enabled', False)
noise_schedule_type = trained_cfg.training.get('latent_noise_schedule', 'cosine')
noise_warmup_epochs = trained_cfg.training.get('latent_noise_warmup_epochs', 0)
total_epochs = trained_cfg.training.epochs
print(f"Noise schedule for this run: enabled={noise_enabled}, shape={noise_schedule_type}, start={noise_start}, "
      f"end={noise_end}, warmup_epochs={noise_warmup_epochs}, over {total_epochs} epochs")

# Re-derive the same train split this run used (same random_seed/subset/split
# ratios, from the run's own saved config) so class_names/train_labels below
# match rep.z's row order exactly -- same pattern dgd_test_inference.ipynb uses.
trained_cfg.data.root_dir = str(project_root / "data")
train_loader, val_loader, test_loader, class_names = create_dataloaders(trained_cfg)
train_labels = collect_all_labels(train_loader)

# Same points report.py's post-hoc figures track for this run (deterministic
# given train_labels/n_classes/random_seed -- see pick_class_representative_indices),
# so this notebook's plots line up with the ones already written to
# figures/training/ under this run_dir.
track_indices = pick_class_representative_indices(train_labels, len(class_names), trained_cfg.random_seed)
track_labels = train_labels[track_indices]


## The noise schedule

$$
D_t = D_{\text{end}} + (D_{\text{start}} - D_{\text{end}}) \cdot \frac{1 + \cos(\pi p_t)}{2}, \qquad p_t = \mathrm{clip}\!\left(\frac{t-1}{T-1},\, 0,\, 1\right)
$$

The formula above is the `cosine` shape -- check the `shape=...` printout in the cell above for what this particular run actually used. `constant` stays flat at `noise_start`, `linear` interpolates straight-line, and `warmup_cosine` holds at 0 for `latent_noise_warmup_epochs` then runs the cosine formula over the rest. All shapes are computed by the same `noise_schedule()` dispatcher (`src/utils/schedules.py`) so this notebook and training can never disagree on what ran. $D_t$ is the target displacement (see the mechanism note above); the actual per-coordinate std used to draw noise is $D_t/\sqrt{d}$. Dots mark the checkpoints visualized below.

In [ ]:
epochs_range = list(range(0, total_epochs + 1))
schedule = [noise_schedule(e, total_epochs, noise_start, noise_end, schedule_type=noise_schedule_type,
                            warmup_steps=noise_warmup_epochs) if noise_enabled else 0.0
            for e in epochs_range]
checkpoint_targets = [noise_schedule(e, total_epochs, noise_start, noise_end, schedule_type=noise_schedule_type,
                                      warmup_steps=noise_warmup_epochs) if noise_enabled else 0.0
                     for e in checkpoint_epochs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, schedule, color='orange', linewidth=2, label='target displacement')
ax.scatter(checkpoint_epochs, checkpoint_targets, color='black', zorder=5, label='saved checkpoints')
for e, s in zip(checkpoint_epochs, checkpoint_targets):
    ax.annotate(f"epoch {e}\ntarget={s:.4f}", (e, s), textcoords="offset points",
                xytext=(0, 10), fontsize=8, ha='center')
ax.set_xlabel('Epoch')
ax.set_ylabel('Target displacement')
ax.set_title(f'Noise Schedule ({noise_schedule_type}) -- {run_dir.name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## What the noise actually does to the representations

For each checkpoint, `plot_noise_comparison` shows two views in a fresh PCA projection of that checkpoint's train representations (PCA, not UMAP/t-SNE, since only a linear projection makes displacement in the picture faithful to displacement in the real space):

- **Left** -- clean $z$ (blue) vs. $\tilde z$ with that checkpoint's noise (red). Red spilling into the gaps between clusters means noise is filling that space.
- **Right** -- one representative point per class (`track_indices`, same points across every checkpoint), each with 100 freshly-sampled noise draws scattered around it, color-coded by class. Illustrative only -- these are sampled fresh at plot time, not the literal draws that epoch's training step used. Annotated with the target displacement $D_t$, the per-coordinate std $D_t/\sqrt{d}$, the population's median nearest-neighbor distance, and their **ratio** ($D_t$ / NN distance) -- the number worth tracking *across* checkpoints, since PCA is refit independently each time and its axes can rotate between epochs.

Right after it, `plot_noise_optimization_shift` shows something different for the same points: not illustrative sampled noise, but the **actual** clean → noised → post-optimization-step positions that checkpoint's real training epoch produced (star → circle → X), read back from `z_trajectory.pth`. This is the direct test of "does noise durably move a representation, or does the optimizer just undo it?" -- watch whether the X lands back near the star (undone) or stays out near/past the circle (persists).

**What to look for:** ratio well above 1 early (noise spans neighboring clusters), dropping well below 1 by the end (noise ball stays inside one cluster's own density).

(Epoch 0 starts every representation at the same point under `distribution: "zeros"`, so PCA variance is `nan%` and NN distance is `0` -- expected for a zero-variance start, not a bug. It also has no z_trajectory.pth -- training hasn't run yet -- so only the noise-comparison plot appears for it, not the shift plot.)

In [ ]:
for epoch, ckpt_dir in zip(checkpoint_epochs, checkpoint_dirs):
    checkpoint = load_checkpoint(ckpt_dir, decoder_factory, device=device)
    rep = checkpoint['rep']
    target_displacement = noise_schedule(epoch, total_epochs, noise_start, noise_end,
                                          schedule_type=noise_schedule_type,
                                          warmup_steps=noise_warmup_epochs) if noise_enabled else 0.0

    plot_noise_comparison(
        representations=rep.z.detach(),
        noise_scale=target_displacement,
        point_indices=track_indices.tolist(),
        point_labels=track_labels.tolist(),
        class_names=class_names,
        title=f"Train Representations -- epoch {epoch} (target={target_displacement:.4f})",
        show=True,
    )

    trajectory_path = ckpt_dir / "z_trajectory.pth"
    if trajectory_path.is_file():
        trajectory = torch.load(trajectory_path, map_location="cpu")
        plot_noise_optimization_shift(
            representations=rep.z.detach(), labels=train_labels, class_names=class_names,
            track_labels=trajectory['train']['labels'],
            z_clean=trajectory['train']['z_clean'], z_noised=trajectory['train']['z_noised'],
            z_after=trajectory['train']['z_after'],
            title=f"Train Representations -- epoch {epoch}",
            show=True,
        )


## Practical takeaway for tuning `noise_start` / `noise_end`

Change `training.latent_noise_*` in `config/config.yaml`, run a fresh training run, then re-run this notebook and watch the `ratio` across checkpoints:

- Ratio never exceeds ~1 anywhere → noise probably isn't spreading representations far enough → raise `latent_noise_start`.
- Ratio still well above 1 at the final checkpoint → representations haven't settled → lower `latent_noise_end`, extend training, or slow the anneal.

The same figures are also generated automatically as part of post-hoc figure generation, gated by `training.plot_interval` (training -- `null` by default, so no intermediate checkpoints exist unless you set it, e.g. to 10, to checkpoint and later plot every 10th epoch) and `training.inference.plot_interval` (the same idea per-step, for `dgd_test_inference.ipynb`'s Algorithm-2 optimization): `figures/training/noise_train/epoch*.png` / `noise_val/epoch*.png` for training (three panels -- the two illustrative ones below, plus a third folding in the real clean/noised/after-optimizer-step positions whenever `z_trajectory.pth` exists for that checkpoint) and `figures/training/noise_test/step*.png` for inference (two panels only -- inference has no equivalent trajectory capture), each stitched into that same subfolder's own `progression.gif`/`.mp4`. This notebook is just a narrated walkthrough of the same plots.